# Notebook 05: Explainable Boosting Machine Analysis

## Purpose

Fit strictly additive Explainable Boosting Machines using the exact folds from Notebook 04, generate out-of-fold probabilities, extract full-sample shape functions, and compare nonlinear predictor relationships across targets and with logistic regression.

## Inputs

- Labelled complete-case data and fixed folds from Notebooks 02 and 04
- Saved full-sample logistic models and metadata from Notebook 04

## Outputs

- EBM out-of-fold probabilities and fitted models
- Feature-importance and shape-function tables
- EBM interpretation figures
- `data/processed/ebm_analysis_metadata.json`

## Dependencies

Run Notebooks 01--04 first. Notebooks 06--09 use the EBM predictions and models.

> **Repository policy:** Notebook outputs and execution counts are cleared in the public source files. Run the notebooks in the documented order to regenerate all results.

## EBM model form

With interactions disabled, each fitted model has the form

$$
\operatorname{logit}\{P(Y=1\mid X)\}
=
\beta_0+\sum_{j=1}^{p}f_j(X_j).
$$

Each learned function \(f_j\) is directly inspectable. The shape functions describe modelled associations with the selected target; they are not causal effects.

Official references:

- Nori, H., Jenkins, S., Koch, P., & Caruana, R. (2019). *InterpretML: A Unified Framework for Machine Learning Interpretability*. arXiv:1909.09223.
- InterpretML documentation: `ExplainableBoostingClassifier`.
- Lou, Y., Caruana, R., Gehrke, J., & Hooker, G. (2013). *Accurate Intelligible Models with Pairwise Interactions*. KDD.

## Software requirement

This notebook targets **InterpretML 0.7.8**, the version used when the notebook was prepared. Install it in the active environment before running:

```bash
python -m pip install "interpret==0.7.8"
```

A different InterpretML version is recorded in the metadata and triggers a warning rather than silently changing the saved specification.

## 1. Setup

In [ ]:
from pathlib import Path
import hashlib
import inspect
import json
import platform
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

from packaging.version import Version

try:
    import interpret
    from interpret.glassbox import ExplainableBoostingClassifier
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "InterpretML is required for Notebook 05. Install the pinned version "
        'with: python -m pip install "interpret==0.7.8"'
    ) from exc

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_rows", 300)
pd.set_option("display.width", 190)


## 2. Fixed analysis configuration

In [ ]:
RANDOM_STATE = 26
N_SPLITS = 5

EXPECTED_INTERPRET_VERSION = "0.7.8"

CONTINUOUS_PREDICTORS = [
    "age",
    "bmi",
    "income_poverty_ratio",
]

CATEGORICAL_PREDICTORS = [
    "sex",
    "race_ethnicity",
    "insurance_history",
]

PREDICTOR_COLUMNS = (
    CONTINUOUS_PREDICTORS
    + CATEGORICAL_PREDICTORS
)

FEATURE_TYPES = [
    "continuous",
    "continuous",
    "continuous",
    "nominal",
    "nominal",
    "nominal",
]

TARGET_COLUMNS = [
    "self_reported_prior_diagnosis",
    "current_hba1c_ge_6_5",
]

TARGET_DISPLAY_NAMES = {
    "self_reported_prior_diagnosis": (
        "Prior reported clinician diagnosis"
    ),
    "current_hba1c_ge_6_5": (
        "Current HbA1c at least 6.5%"
    ),
}

TARGET_SHORT_NAMES = {
    "self_reported_prior_diagnosis": "Prior diagnosis",
    "current_hba1c_ge_6_5": "HbA1c ≥ 6.5%",
}

OOF_PROBABILITY_COLUMNS = {
    "self_reported_prior_diagnosis": (
        "ebm_oof_probability_prior_diagnosis"
    ),
    "current_hba1c_ge_6_5": (
        "ebm_oof_probability_hba1c_ge_6_5"
    ),
}

SEX_LEVELS = [
    "Female",
    "Male",
]

RACE_ETHNICITY_LEVELS = [
    "Mexican American",
    "Other Hispanic",
    "Non-Hispanic White",
    "Non-Hispanic Black",
    "Non-Hispanic Asian",
    "Other or multiracial",
]

INSURANCE_HISTORY_LEVELS = [
    "Continuously insured",
    "Currently insured, past-year gap",
    "Currently uninsured",
]

CATEGORY_LEVELS = {
    "sex": SEX_LEVELS,
    "race_ethnicity": RACE_ETHNICITY_LEVELS,
    "insurance_history": INSURANCE_HISTORY_LEVELS,
}

REFERENCE_CATEGORIES = {
    "sex": "Female",
    "race_ethnicity": "Non-Hispanic White",
    "insurance_history": "Continuously insured",
}

CONTINUOUS_REFERENCE_VALUES = {
    "age": 40.0,
    "bmi": 25.0,
    "income_poverty_ratio": 1.0,
}

CONTINUOUS_DISPLAY_NAMES = {
    "age": "Age in years",
    "bmi": "Body mass index",
    "income_poverty_ratio": "Income-to-poverty ratio",
}

CATEGORICAL_DISPLAY_NAMES = {
    "sex": "Sex",
    "race_ethnicity": "Race/ethnicity",
    "insurance_history": "Insurance history",
}

JOINT_LABEL_ORDER = [
    "D0_H0",
    "D0_H1",
    "D1_H0",
    "D1_H1",
]

# The complete primary EBM specification is fixed and reused for both targets.
EBM_PARAMETERS = {
    "feature_names": PREDICTOR_COLUMNS,
    "feature_types": FEATURE_TYPES,
    "max_bins": 1024,
    "max_interaction_bins": 64,
    "interactions": 0,
    "exclude": None,
    "validation_size": 0.15,
    "outer_bags": 14,
    "inner_bags": 0,
    "learning_rate": 0.015,
    "greedy_ratio": 10.0,
    "cyclic_progress": False,
    "smoothing_rounds": 75,
    "interaction_smoothing_rounds": 75,
    "max_rounds": 50000,
    "early_stopping_rounds": 100,
    "early_stopping_tolerance": 1e-5,
    "min_samples_leaf": 4,
    "min_hessian": 1e-4,
    "reg_alpha": 0.0,
    "reg_lambda": 0.0,
    "max_delta_step": 0.0,
    "gain_scale": 5.0,
    "min_cat_samples": 10,
    "cat_smooth": 10.0,
    "missing": "separate",
    "max_leaves": 2,
    "monotone_constraints": None,
    "objective": "log_loss",
    "n_jobs": -2,
    "random_state": RANDOM_STATE,
}

if interpret.__version__ != EXPECTED_INTERPRET_VERSION:
    warnings.warn(
        "This notebook was prepared for InterpretML "
        f"{EXPECTED_INTERPRET_VERSION}, but the active version is "
        f"{interpret.__version__}. The fitted version will be recorded in "
        "metadata. Review any version-dependent differences before final use.",
        stacklevel=1,
    )

if Version(interpret.__version__) < Version("0.7.0"):
    raise RuntimeError(
        "InterpretML 0.7.0 or newer is required by this notebook. "
        'Install the pinned version with: python -m pip install "interpret==0.7.8"'
    )

constructor_parameters = set(
    inspect.signature(
        ExplainableBoostingClassifier
    ).parameters
)

unsupported_parameters = set(
    EBM_PARAMETERS
).difference(constructor_parameters)

if unsupported_parameters:
    raise RuntimeError(
        "The active InterpretML version does not support the fixed EBM "
        f"parameters: {sorted(unsupported_parameters)}"
    )

print("InterpretML version:", interpret.__version__)
print("Interactions:", EBM_PARAMETERS["interactions"])
print("Outer bags:", EBM_PARAMETERS["outer_bags"])


### Why these EBM settings are fixed

The same configuration is applied to both targets so that target comparisons are not confounded by target-specific tuning.

- `interactions=0` enforces a purely additive model.
- Internal validation and early stopping occur **inside each outer training fold**.
- Outer bagging stabilises the learned shape functions.
- Continuous predictors are not standardised because EBM learns functions on their original scales.
- Categorical variables are passed as nominal features.
- No class weighting or survey weighting is used in the primary predictive models.
- Race/ethnicity remains a predictor in this primary specification.

## 3. Project paths

In [ ]:
PROJECT_DIR = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
OUTPUT_DIR = PROJECT_DIR / "outputs"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"
MODEL_DIR = OUTPUT_DIR / "models"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

LABELLED_DATA_PATH = (
    PROCESSED_DIR
    / "nhanes_diabetes_complete_case_labeled.csv"
)

FOLD_ASSIGNMENT_PATH = (
    PROCESSED_DIR / "primary_cv_fold_assignments.csv"
)

SAMPLE_METADATA_PATH = (
    PROCESSED_DIR / "sample_metadata.json"
)

SAMPLE_LABEL_METADATA_PATH = (
    PROCESSED_DIR / "sample_and_label_metadata.json"
)

LOGISTIC_METADATA_PATH = (
    PROCESSED_DIR / "logistic_regression_metadata.json"
)

LOGISTIC_MODEL_PATHS = {
    "self_reported_prior_diagnosis": (
        MODEL_DIR
        / "logistic_self_reported_prior_diagnosis.joblib"
    ),
    "current_hba1c_ge_6_5": (
        MODEL_DIR
        / "logistic_current_hba1c_ge_6_5.joblib"
    ),
}

EBM_OOF_PREDICTIONS_PATH = (
    PROCESSED_DIR / "ebm_oof_predictions.csv"
)

EBM_METADATA_PATH = (
    PROCESSED_DIR / "ebm_analysis_metadata.json"
)

EBM_MODEL_PATHS = {
    "self_reported_prior_diagnosis": (
        MODEL_DIR
        / "ebm_self_reported_prior_diagnosis.joblib"
    ),
    "current_hba1c_ge_6_5": (
        MODEL_DIR
        / "ebm_current_hba1c_ge_6_5.joblib"
    ),
}

EBM_JSON_PATHS = {
    "self_reported_prior_diagnosis": (
        MODEL_DIR
        / "ebm_self_reported_prior_diagnosis.json"
    ),
    "current_hba1c_ge_6_5": (
        MODEL_DIR
        / "ebm_current_hba1c_ge_6_5.json"
    ),
}

required_paths = [
    LABELLED_DATA_PATH,
    FOLD_ASSIGNMENT_PATH,
    SAMPLE_METADATA_PATH,
    SAMPLE_LABEL_METADATA_PATH,
    LOGISTIC_METADATA_PATH,
    *LOGISTIC_MODEL_PATHS.values(),
]

missing_paths = [
    path for path in required_paths
    if not path.exists()
]

if missing_paths:
    missing_text = "\n".join(
        f"- {path}" for path in missing_paths
    )

    raise FileNotFoundError(
        "Notebook 05 requires completed outputs from Notebooks 01–04. "
        "The following files are missing:\n"
        f"{missing_text}"
    )

print("Project directory:", PROJECT_DIR)
print("Input data:", LABELLED_DATA_PATH)
print("Fixed folds:", FOLD_ASSIGNMENT_PATH)


## 4. Load data, folds, metadata, and logistic models

In [ ]:
data = pd.read_csv(LABELLED_DATA_PATH)
fold_assignments = pd.read_csv(FOLD_ASSIGNMENT_PATH)

with SAMPLE_METADATA_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    sample_metadata = json.load(file)

with SAMPLE_LABEL_METADATA_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    sample_label_metadata = json.load(file)

with LOGISTIC_METADATA_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    logistic_metadata = json.load(file)

logistic_models = {
    target: joblib.load(path)
    for target, path in LOGISTIC_MODEL_PATHS.items()
}

print("Loaded complete-case data:", data.shape)
print("Loaded fold assignments:", fold_assignments.shape)
print("Loaded two logistic models for functional-form comparison.")


## 5. Validate that Notebook 05 uses the exact Notebook 04 sample and folds

In [ ]:
required_data_columns = set(
    [
        "id",
        "joint_label_code",
        "label_group",
        "confirmed_current_pregnancy",
        "primary_sample_eligible",
        "exam_status",
        "phlebotomy_weight",
        "survey_stratum",
        "survey_psu",
    ]
    + PREDICTOR_COLUMNS
    + TARGET_COLUMNS
)

missing_data_columns = required_data_columns.difference(
    data.columns
)

if missing_data_columns:
    raise KeyError(
        "The modelling dataset is missing columns: "
        f"{sorted(missing_data_columns)}"
    )

required_fold_columns = {
    "id",
    "joint_label_code",
    "self_reported_prior_diagnosis",
    "current_hba1c_ge_6_5",
    "cv_fold",
}

missing_fold_columns = required_fold_columns.difference(
    fold_assignments.columns
)

if missing_fold_columns:
    raise KeyError(
        "The fold file is missing columns: "
        f"{sorted(missing_fold_columns)}"
    )

data = (
    data
    .sort_values("id")
    .reset_index(drop=True)
)

fold_assignments = (
    fold_assignments
    .sort_values("id")
    .reset_index(drop=True)
)

integer_columns = [
    "id",
    "self_reported_prior_diagnosis",
    "current_hba1c_ge_6_5",
    "confirmed_current_pregnancy",
    "primary_sample_eligible",
    "exam_status",
    "survey_stratum",
    "survey_psu",
]

for column in integer_columns:
    data[column] = pd.to_numeric(
        data[column],
        errors="raise",
    ).astype("Int64")

fold_assignments["id"] = pd.to_numeric(
    fold_assignments["id"],
    errors="raise",
).astype("Int64")

fold_assignments["cv_fold"] = pd.to_numeric(
    fold_assignments["cv_fold"],
    errors="raise",
).astype(int)

for column in CONTINUOUS_PREDICTORS:
    data[column] = pd.to_numeric(
        data[column],
        errors="raise",
    ).astype(float)

for column in CATEGORICAL_PREDICTORS:
    data[column] = (
        data[column]
        .astype("string")
    )

data["joint_label_code"] = (
    data["joint_label_code"]
    .astype("string")
)

if not data["id"].is_unique:
    raise ValueError(
        "Participant IDs are not unique."
    )

if not fold_assignments["id"].is_unique:
    raise ValueError(
        "The fold file contains duplicate participant IDs."
    )

if set(data["id"].astype(int)) != set(
    fold_assignments["id"].astype(int)
):
    raise ValueError(
        "The fixed fold file belongs to a different analytic sample."
    )

comparison_columns = [
    "joint_label_code",
    *TARGET_COLUMNS,
]

for column in comparison_columns:
    if not (
        data[column].astype(str)
        .eq(
            fold_assignments[column].astype(str)
        )
        .all()
    ):
        raise ValueError(
            "The fixed folds disagree with the current data in "
            f"{column}."
        )

data = data.merge(
    fold_assignments[["id", "cv_fold"]],
    on="id",
    how="left",
    validate="one_to_one",
)

if data["cv_fold"].isna().any():
    raise ValueError(
        "At least one participant has no fixed fold assignment."
    )

data["cv_fold"] = data["cv_fold"].astype(int)

if set(data["cv_fold"]) != set(
    range(1, N_SPLITS + 1)
):
    raise ValueError(
        "The saved assignment does not contain exactly folds 1–5."
    )

model_required_columns = (
    PREDICTOR_COLUMNS
    + TARGET_COLUMNS
    + [
        "joint_label_code",
        "cv_fold",
        "confirmed_current_pregnancy",
        "primary_sample_eligible",
    ]
)

if data[model_required_columns].isna().sum().sum() != 0:
    raise ValueError(
        "The modelling data contain missing required values."
    )

if data["confirmed_current_pregnancy"].ne(0).any():
    raise ValueError(
        "The modelling data contain a confirmed current pregnancy."
    )

if not data["primary_sample_eligible"].eq(1).all():
    raise ValueError(
        "The modelling data contain an ineligible participant."
    )

if not data["exam_status"].eq(2).all():
    raise ValueError(
        "At least one participant did not complete the MEC examination."
    )

if not data["phlebotomy_weight"].gt(0).all():
    raise ValueError(
        "At least one participant has a non-positive phlebotomy weight."
    )

for target in TARGET_COLUMNS:
    observed_target_values = set(
        data[target].astype(int).unique()
    )

    if not observed_target_values.issubset({0, 1}):
        raise ValueError(
            f"{target} contains values other than 0 and 1."
        )

expected_joint_code = (
    "D"
    + data["self_reported_prior_diagnosis"].astype(str)
    + "_H"
    + data["current_hba1c_ge_6_5"].astype(str)
)

if not expected_joint_code.eq(
    data["joint_label_code"]
).all():
    raise ValueError(
        "The joint-label code disagrees with the two targets."
    )

for variable, allowed_levels in CATEGORY_LEVELS.items():
    observed_levels = set(
        data[variable].dropna().astype(str).unique()
    )

    unexpected_levels = observed_levels.difference(
        allowed_levels
    )

    if unexpected_levels:
        raise ValueError(
            f"{variable} contains unexpected categories: "
            f"{sorted(unexpected_levels)}"
        )

    absent_levels = set(allowed_levels).difference(
        observed_levels
    )

    if absent_levels:
        raise ValueError(
            f"{variable} is missing prespecified categories: "
            f"{sorted(absent_levels)}"
        )

placeholder_columns = [
    "income_poverty_ratio",
    "bmi",
    "phlebotomy_weight",
]

for column in placeholder_columns:
    placeholder_mask = (
        data[column].notna()
        & data[column].gt(0)
        & data[column].lt(1e-50)
    )

    if placeholder_mask.any():
        raise ValueError(
            f"{column} still contains an SAS/XPT numeric "
            "missing-value placeholder."
        )

if len(data) != int(sample_metadata["n_complete_case"]):
    raise ValueError(
        "The analytic sample size does not match Notebook 01 metadata."
    )

if len(data) != int(
    sample_label_metadata["complete_case_n"]
):
    raise ValueError(
        "The analytic sample size does not match Notebook 02 metadata."
    )

analysis_id_hash = hashlib.sha256(
    ",".join(
        data["id"].astype(str).tolist()
    ).encode("utf-8")
).hexdigest()

if analysis_id_hash != logistic_metadata[
    "analysis_id_sha256"
]:
    raise ValueError(
        "Notebook 05 and Notebook 04 do not use the same participants."
    )

if logistic_metadata["random_state"] != RANDOM_STATE:
    raise ValueError(
        "Notebook 04 used a different random seed."
    )

if logistic_metadata["n_splits"] != N_SPLITS:
    raise ValueError(
        "Notebook 04 used a different number of folds."
    )

if logistic_metadata["predictor_columns"] != PREDICTOR_COLUMNS:
    raise ValueError(
        "Notebook 04 used a different predictor set or order."
    )

if logistic_metadata["target_columns"] != TARGET_COLUMNS:
    raise ValueError(
        "Notebook 04 used a different target order."
    )

validation_summary = {
    "analytic_sample_n": int(len(data)),
    "analysis_id_sha256": analysis_id_hash,
    "five_folds_present": bool(
        set(data["cv_fold"])
        == set(range(1, N_SPLITS + 1))
    ),
    "confirmed_current_pregnancy_n": int(
        data["confirmed_current_pregnancy"].sum()
    ),
    "prior_diagnosis_positive_n": int(
        data["self_reported_prior_diagnosis"].sum()
    ),
    "hba1c_positive_n": int(
        data["current_hba1c_ge_6_5"].sum()
    ),
}

print("Cross-notebook validation passed.")
validation_summary


## 6. Prepare the EBM predictor matrix

EBM receives continuous variables on their original scales and categorical variables as nominal strings.

In [ ]:
def prepare_ebm_X(
    dataframe: pd.DataFrame,
) -> pd.DataFrame:
    X = dataframe[
        PREDICTOR_COLUMNS
    ].copy()

    for variable in CONTINUOUS_PREDICTORS:
        X[variable] = pd.to_numeric(
            X[variable],
            errors="raise",
        ).astype(float)

    for variable in CATEGORICAL_PREDICTORS:
        X[variable] = (
            X[variable]
            .astype(str)
        )

    return X


X_all = prepare_ebm_X(data)

if X_all.isna().sum().sum() != 0:
    raise ValueError(
        "The EBM predictor matrix contains missing values."
    )

X_all.dtypes


## 7. Verify category coverage inside every fixed fold

Every training fold must contain every prespecified category so that no held-out category is unseen during fitting.

In [ ]:
category_fold_rows = []

for fold in range(1, N_SPLITS + 1):
    training_data = data.loc[
        data["cv_fold"] != fold
    ]
    validation_data = data.loc[
        data["cv_fold"] == fold
    ]

    for variable, levels in CATEGORY_LEVELS.items():
        for level in levels:
            training_n = int(
                training_data[variable]
                .astype(str)
                .eq(level)
                .sum()
            )

            validation_n = int(
                validation_data[variable]
                .astype(str)
                .eq(level)
                .sum()
            )

            category_fold_rows.append(
                {
                    "cv_fold": fold,
                    "variable": variable,
                    "level": level,
                    "training_n": training_n,
                    "validation_n": validation_n,
                }
            )

category_fold_coverage = pd.DataFrame(
    category_fold_rows
)

if category_fold_coverage["training_n"].eq(0).any():
    problematic = category_fold_coverage.loc[
        category_fold_coverage["training_n"].eq(0)
    ]

    raise ValueError(
        "At least one category is absent from an EBM training fold:\n"
        f"{problematic}"
    )

category_fold_coverage.to_csv(
    TABLE_DIR / "ebm_category_fold_coverage.csv",
    index=False,
)

category_fold_coverage


## Primary EBM fitting

## 8. EBM construction and validation helpers

In [ ]:
def build_ebm() -> ExplainableBoostingClassifier:
    return ExplainableBoostingClassifier(
        **EBM_PARAMETERS
    )


def validate_additive_ebm(
    model: ExplainableBoostingClassifier,
) -> None:
    if model.interactions != 0:
        raise ValueError(
            "The fitted EBM was not configured with interactions=0."
        )

    if any(
        len(term_features) != 1
        for term_features in model.term_features_
    ):
        raise ValueError(
            "The fitted EBM contains a non-additive interaction term."
        )

    if list(model.feature_names_in_) != PREDICTOR_COLUMNS:
        raise ValueError(
            "The fitted EBM feature order differs from the "
            "prespecified predictor order."
        )

    if list(model.term_names_) != PREDICTOR_COLUMNS:
        raise ValueError(
            "The fitted EBM terms differ from the six main effects."
        )

    if list(model.feature_types_in_) != FEATURE_TYPES:
        raise ValueError(
            "The fitted EBM feature types differ from the "
            "prespecified continuous/nominal types."
        )

    if list(model.classes_) != [0, 1]:
        raise ValueError(
            "The fitted EBM class order is not [0, 1]."
        )


def main_stage_iteration_summary(
    model: ExplainableBoostingClassifier,
) -> dict:
    best_iteration = np.asarray(
        model.best_iteration_
    )

    if best_iteration.ndim != 2:
        raise ValueError(
            "Unexpected EBM best_iteration_ structure."
        )

    main_iterations = (
        best_iteration[0]
        .astype(float)
    )

    return {
        "best_iteration_min": int(
            np.min(main_iterations)
        ),
        "best_iteration_median": float(
            np.median(main_iterations)
        ),
        "best_iteration_max": int(
            np.max(main_iterations)
        ),
    }


## 9. Generate out-of-fold EBM probabilities

The fixed fold file from Notebook 04 is loaded rather than recreated. Internal EBM validation is confined to the outer training data.

In [ ]:
oof_predictions = data[
    [
        "id",
        "cv_fold",
        "joint_label_code",
        *TARGET_COLUMNS,
    ]
].copy()

fold_fit_rows = []
fold_importance_rows = []

oof_start_time = time.time()

for target in TARGET_COLUMNS:
    probability_column = (
        OOF_PROBABILITY_COLUMNS[target]
    )

    oof_predictions[
        probability_column
    ] = np.nan

    for fold in range(1, N_SPLITS + 1):
        validation_mask = (
            data["cv_fold"] == fold
        )
        training_mask = ~validation_mask

        X_train = prepare_ebm_X(
            data.loc[training_mask]
        )
        X_valid = prepare_ebm_X(
            data.loc[validation_mask]
        )

        y_train = (
            data.loc[training_mask, target]
            .astype(int)
        )
        y_valid = (
            data.loc[validation_mask, target]
            .astype(int)
        )

        if y_train.nunique() != 2:
            raise ValueError(
                f"Training fold {fold} has only one class for {target}."
            )

        model = build_ebm()
        fit_start = time.time()

        model.fit(
            X_train,
            y_train,
            sample_weight=None,
        )

        fit_seconds = (
            time.time() - fit_start
        )

        validate_additive_ebm(model)

        validation_probability = (
            model.predict_proba(
                X_valid
            )[:, 1]
        )

        if not np.isfinite(
            validation_probability
        ).all():
            raise ValueError(
                f"Non-finite EBM probabilities for {target}, fold {fold}."
            )

        if not (
            (0 <= validation_probability)
            & (validation_probability <= 1)
        ).all():
            raise ValueError(
                f"EBM probabilities outside [0, 1] for {target}, fold {fold}."
            )

        oof_predictions.loc[
            validation_mask,
            probability_column,
        ] = validation_probability

        iteration_summary = (
            main_stage_iteration_summary(
                model
            )
        )

        fold_fit_rows.append(
            {
                "target": target,
                "target_display_name": (
                    TARGET_DISPLAY_NAMES[target]
                ),
                "cv_fold": fold,
                "training_n": int(
                    training_mask.sum()
                ),
                "validation_n": int(
                    validation_mask.sum()
                ),
                "training_positive_n": int(
                    y_train.sum()
                ),
                "validation_positive_n": int(
                    y_valid.sum()
                ),
                "training_positive_share": float(
                    y_train.mean()
                ),
                "validation_positive_share": float(
                    y_valid.mean()
                ),
                "fit_seconds": float(
                    fit_seconds
                ),
                "n_main_terms": int(
                    len(model.term_names_)
                ),
                "n_interaction_terms": int(
                    sum(
                        len(term) > 1
                        for term in model.term_features_
                    )
                ),
                "minimum_validation_probability": float(
                    validation_probability.min()
                ),
                "maximum_validation_probability": float(
                    validation_probability.max()
                ),
                **iteration_summary,
            }
        )

        importances = model.term_importances(
            importance_type="avg_weight"
        )

        for term, importance in zip(
            model.term_names_,
            importances,
        ):
            fold_importance_rows.append(
                {
                    "target": target,
                    "cv_fold": fold,
                    "term": term,
                    "importance": float(
                        importance
                    ),
                }
            )

        elapsed_minutes = (
            time.time() - oof_start_time
        ) / 60

        print(
            f"Completed target={target}, fold={fold} "
            f"({fit_seconds:.1f} seconds; "
            f"{elapsed_minutes:.1f} total minutes)."
        )

for target in TARGET_COLUMNS:
    probability_column = (
        OOF_PROBABILITY_COLUMNS[target]
    )

    if oof_predictions[
        probability_column
    ].isna().any():
        raise ValueError(
            f"Missing out-of-fold EBM probabilities for {target}."
        )

    if not oof_predictions[
        probability_column
    ].between(0, 1).all():
        raise ValueError(
            f"Out-of-fold EBM probabilities for {target} "
            "lie outside [0, 1]."
        )

fold_fit_summary = pd.DataFrame(
    fold_fit_rows
)

fold_importances = pd.DataFrame(
    fold_importance_rows
)

oof_predictions.to_csv(
    EBM_OOF_PREDICTIONS_PATH,
    index=False,
)

fold_fit_summary.to_csv(
    TABLE_DIR / "ebm_fold_fit_summary.csv",
    index=False,
)

fold_importances.to_csv(
    TABLE_DIR / "ebm_fold_feature_importance.csv",
    index=False,
)

print("Saved EBM out-of-fold predictions to:")
print(EBM_OOF_PREDICTIONS_PATH)

fold_fit_summary


Performance metrics are deliberately deferred to Notebook 06, where logistic regression and EBM will be evaluated with identical metric definitions.

## 10. Fit and save full-sample EBMs for interpretation

In [ ]:
full_ebm_models = {}
full_fit_rows = []

for target in TARGET_COLUMNS:
    y = data[target].astype(int)

    model = build_ebm()
    fit_start = time.time()

    model.fit(
        X_all,
        y,
        sample_weight=None,
    )

    fit_seconds = (
        time.time() - fit_start
    )

    validate_additive_ebm(model)

    full_ebm_models[target] = model

    joblib.dump(
        model,
        EBM_MODEL_PATHS[target],
    )

    model.to_json(
        EBM_JSON_PATHS[target],
        detail="interpretable",
        indent=2,
    )

    full_probability = (
        model.predict_proba(
            X_all
        )[:, 1]
    )

    iteration_summary = (
        main_stage_iteration_summary(
            model
        )
    )

    full_fit_rows.append(
        {
            "target": target,
            "target_display_name": (
                TARGET_DISPLAY_NAMES[target]
            ),
            "analytic_n": int(
                len(data)
            ),
            "positive_n": int(
                y.sum()
            ),
            "positive_share": float(
                y.mean()
            ),
            "fit_seconds": float(
                fit_seconds
            ),
            "n_main_terms": int(
                len(model.term_names_)
            ),
            "n_interaction_terms": int(
                sum(
                    len(term) > 1
                    for term in model.term_features_
                )
            ),
            "intercept_log_odds": float(
                np.asarray(
                    model.intercept_
                ).reshape(-1)[0]
            ),
            "minimum_full_sample_probability": float(
                full_probability.min()
            ),
            "maximum_full_sample_probability": float(
                full_probability.max()
            ),
            **iteration_summary,
        }
    )

    print(
        f"Saved full-sample EBM for {target}:"
    )
    print("-", EBM_MODEL_PATHS[target])
    print("-", EBM_JSON_PATHS[target])

full_fit_summary = pd.DataFrame(
    full_fit_rows
)

full_fit_summary.to_csv(
    TABLE_DIR / "ebm_full_fit_summary.csv",
    index=False,
)

full_fit_summary


## Global feature importance

## 11. Full-sample EBM importance

InterpretML's weighted average absolute term contribution is reported. Importance is descriptive and target-specific; a larger value does not imply a causal effect or biological importance.

In [ ]:
FEATURE_DISPLAY_NAMES = {
    "age": "Age",
    "bmi": "BMI",
    "income_poverty_ratio": (
        "Income-to-poverty ratio"
    ),
    "sex": "Sex",
    "race_ethnicity": "Race/ethnicity",
    "insurance_history": "Insurance history",
}

importance_rows = []

for target, model in full_ebm_models.items():
    importances = model.term_importances(
        importance_type="avg_weight"
    )

    total_importance = float(
        np.sum(importances)
    )

    if total_importance <= 0:
        raise ValueError(
            f"Total EBM importance is non-positive for {target}."
        )

    for term, importance in zip(
        model.term_names_,
        importances,
    ):
        importance_rows.append(
            {
                "target": target,
                "target_display_name": (
                    TARGET_DISPLAY_NAMES[target]
                ),
                "term": term,
                "display_name": (
                    FEATURE_DISPLAY_NAMES[term]
                ),
                "importance_mean_absolute_log_odds": float(
                    importance
                ),
                "normalised_importance_share": float(
                    importance / total_importance
                ),
            }
        )

global_feature_importance = pd.DataFrame(
    importance_rows
)

global_feature_importance[
    "within_target_rank"
] = (
    global_feature_importance
    .groupby("target")[
        "importance_mean_absolute_log_odds"
    ]
    .rank(
        method="dense",
        ascending=False,
    )
    .astype(int)
)

global_feature_importance.to_csv(
    TABLE_DIR / "ebm_global_feature_importance.csv",
    index=False,
)

global_feature_importance


## 12. Plot global feature importance

In [ ]:
importance_order = (
    global_feature_importance
    .groupby(
        ["term", "display_name"],
        as_index=False,
    )["normalised_importance_share"]
    .mean()
    .sort_values(
        "normalised_importance_share",
        ascending=False,
    )["term"]
    .tolist()
)

importance_name_lookup = (
    global_feature_importance
    .drop_duplicates("term")
    .set_index("term")["display_name"]
    .to_dict()
)

vertical_positions = np.arange(
    len(importance_order)
)

target_offsets = {
    TARGET_COLUMNS[0]: -0.12,
    TARGET_COLUMNS[1]: 0.12,
}

target_markers = {
    TARGET_COLUMNS[0]: "o",
    TARGET_COLUMNS[1]: "s",
}

figure, axis = plt.subplots(
    figsize=(9, 5.5)
)

for target in TARGET_COLUMNS:
    target_data = (
        global_feature_importance
        .loc[
            global_feature_importance["target"]
            == target
        ]
        .set_index("term")
        .reindex(importance_order)
    )

    axis.plot(
        100
        * target_data[
            "normalised_importance_share"
        ].to_numpy(),
        vertical_positions
        + target_offsets[target],
        linestyle="none",
        marker=target_markers[target],
        label=TARGET_SHORT_NAMES[target],
    )

axis.set_yticks(
    vertical_positions,
    labels=[
        importance_name_lookup[term]
        for term in importance_order
    ],
)

axis.set_xlabel(
    "Share of total EBM importance within target (%)"
)
axis.set_ylabel("Predictor")
axis.set_title(
    "EBM global feature importance"
)
axis.legend()
axis.invert_yaxis()

figure.tight_layout()

importance_png_path = (
    FIGURE_DIR
    / "ebm_global_feature_importance.png"
)

importance_pdf_path = (
    FIGURE_DIR
    / "ebm_global_feature_importance.pdf"
)

figure.savefig(
    importance_png_path,
    dpi=300,
    bbox_inches="tight",
)

figure.savefig(
    importance_pdf_path,
    bbox_inches="tight",
)

plt.show()
plt.close(figure)

print("Saved:")
print(importance_png_path)
print(importance_pdf_path)


## Shape-function extraction

## 13. Define common evaluation grids and reference values

The same grid is used for both target models. Continuous curves are centred at prespecified reference values:

- age: 40 years;
- BMI: 25;
- income-to-poverty ratio: 1.

BMI is displayed from the 1st to the 99th percentile to avoid allowing a few sparse extreme observations to dominate the horizontal scale. The original observations remain in model fitting.

In [ ]:
continuous_support_rows = []
continuous_grids = {}

for variable in CONTINUOUS_PREDICTORS:
    values = data[variable].astype(float)

    support = {
        "variable": variable,
        "display_name": (
            CONTINUOUS_DISPLAY_NAMES[variable]
        ),
        "minimum": float(values.min()),
        "q01": float(values.quantile(0.01)),
        "q10": float(values.quantile(0.10)),
        "q25": float(values.quantile(0.25)),
        "median": float(values.median()),
        "q75": float(values.quantile(0.75)),
        "q90": float(values.quantile(0.90)),
        "q99": float(values.quantile(0.99)),
        "maximum": float(values.max()),
        "reference_value": float(
            CONTINUOUS_REFERENCE_VALUES[variable]
        ),
    }

    continuous_support_rows.append(support)

    if variable == "age":
        grid = np.arange(
            int(values.min()),
            int(values.max()) + 1,
            dtype=float,
        )

    elif variable == "bmi":
        grid = np.linspace(
            values.quantile(0.01),
            values.quantile(0.99),
            220,
        )

    elif variable == "income_poverty_ratio":
        grid = np.linspace(
            values.min(),
            values.max(),
            220,
        )

    else:
        raise KeyError(
            f"No grid rule defined for {variable}."
        )

    grid = np.unique(
        np.concatenate(
            [
                grid,
                np.array(
                    [
                        CONTINUOUS_REFERENCE_VALUES[
                            variable
                        ]
                    ],
                    dtype=float,
                ),
            ]
        )
    )

    continuous_grids[variable] = grid

continuous_support = pd.DataFrame(
    continuous_support_rows
)

for row in continuous_support_rows:
    reference = row["reference_value"]

    if not (
        row["minimum"]
        <= reference
        <= row["maximum"]
    ):
        raise ValueError(
            f"The reference value for {row['variable']} "
            "lies outside observed support."
        )

continuous_support.to_csv(
    TABLE_DIR / "ebm_continuous_shape_support.csv",
    index=False,
)

continuous_support


## 14. Baseline frame and term-evaluation helpers

Because the EBM is additive, the contribution of one term does not depend on the values assigned to the other predictors. A valid baseline row is nevertheless supplied to the prediction interface.

In [ ]:
BASELINE_VALUES = {
    "age": float(
        data["age"].median()
    ),
    "bmi": float(
        data["bmi"].median()
    ),
    "income_poverty_ratio": float(
        data["income_poverty_ratio"].median()
    ),
    "sex": REFERENCE_CATEGORIES["sex"],
    "race_ethnicity": (
        REFERENCE_CATEGORIES[
            "race_ethnicity"
        ]
    ),
    "insurance_history": (
        REFERENCE_CATEGORIES[
            "insurance_history"
        ]
    ),
}


def make_evaluation_frame(
    variable: str,
    values,
) -> pd.DataFrame:
    values = list(values)

    evaluation = pd.DataFrame(
        {
            predictor: [
                BASELINE_VALUES[predictor]
            ]
            * len(values)
            for predictor in PREDICTOR_COLUMNS
        }
    )

    evaluation[variable] = values

    return prepare_ebm_X(evaluation)


def ebm_reference_centered_term_scores(
    model: ExplainableBoostingClassifier,
    variable: str,
    values,
    reference_value,
) -> np.ndarray:
    term_index = list(
        model.term_names_
    ).index(variable)

    evaluation = make_evaluation_frame(
        variable,
        values,
    )

    reference_frame = (
        make_evaluation_frame(
            variable,
            [reference_value],
        )
    )

    term_scores = model.eval_terms(
        evaluation
    )[:, term_index]

    reference_score = model.eval_terms(
        reference_frame
    )[0, term_index]

    return (
        term_scores
        - reference_score
    )


def logistic_reference_centered_log_odds(
    model,
    variable: str,
    values,
    reference_value,
) -> np.ndarray:
    evaluation = make_evaluation_frame(
        variable,
        values,
    )

    reference_frame = (
        make_evaluation_frame(
            variable,
            [reference_value],
        )
    )

    evaluation_score = model.decision_function(
        evaluation
    )

    reference_score = model.decision_function(
        reference_frame
    )[0]

    return (
        np.asarray(evaluation_score)
        - float(reference_score)
    )


## 15. Extract continuous EBM and logistic shapes on common grids

In [ ]:
continuous_shape_rows = []

for target in TARGET_COLUMNS:
    ebm_model = full_ebm_models[target]
    logistic_model = logistic_models[target]

    for variable in CONTINUOUS_PREDICTORS:
        grid = continuous_grids[variable]
        reference = (
            CONTINUOUS_REFERENCE_VALUES[
                variable
            ]
        )

        ebm_scores = (
            ebm_reference_centered_term_scores(
                ebm_model,
                variable,
                grid,
                reference,
            )
        )

        logistic_scores = (
            logistic_reference_centered_log_odds(
                logistic_model,
                variable,
                grid,
                reference,
            )
        )

        for value, ebm_score, logistic_score in zip(
            grid,
            ebm_scores,
            logistic_scores,
        ):
            continuous_shape_rows.append(
                {
                    "target": target,
                    "target_display_name": (
                        TARGET_DISPLAY_NAMES[target]
                    ),
                    "variable": variable,
                    "variable_display_name": (
                        CONTINUOUS_DISPLAY_NAMES[
                            variable
                        ]
                    ),
                    "value": float(value),
                    "reference_value": float(
                        reference
                    ),
                    "ebm_reference_centered_log_odds": float(
                        ebm_score
                    ),
                    "logistic_reference_centered_log_odds": float(
                        logistic_score
                    ),
                    "ebm_minus_logistic_log_odds": float(
                        ebm_score
                        - logistic_score
                    ),
                }
            )

continuous_shape_functions = pd.DataFrame(
    continuous_shape_rows
)

continuous_shape_functions.to_csv(
    TABLE_DIR / "ebm_continuous_shape_functions.csv",
    index=False,
)

continuous_shape_functions


## 16. Extract categorical EBM and logistic contributions

Every categorical shape is centred at the same reference category used in Notebook 04.

In [ ]:
categorical_shape_rows = []

for target in TARGET_COLUMNS:
    ebm_model = full_ebm_models[target]
    logistic_model = logistic_models[target]

    for variable in CATEGORICAL_PREDICTORS:
        levels = CATEGORY_LEVELS[variable]
        reference = (
            REFERENCE_CATEGORIES[variable]
        )

        ebm_scores = (
            ebm_reference_centered_term_scores(
                ebm_model,
                variable,
                levels,
                reference,
            )
        )

        logistic_scores = (
            logistic_reference_centered_log_odds(
                logistic_model,
                variable,
                levels,
                reference,
            )
        )

        category_counts = (
            data[variable]
            .astype(str)
            .value_counts()
            .reindex(levels)
        )

        for level, ebm_score, logistic_score in zip(
            levels,
            ebm_scores,
            logistic_scores,
        ):
            categorical_shape_rows.append(
                {
                    "target": target,
                    "target_display_name": (
                        TARGET_DISPLAY_NAMES[target]
                    ),
                    "variable": variable,
                    "variable_display_name": (
                        CATEGORICAL_DISPLAY_NAMES[
                            variable
                        ]
                    ),
                    "level": level,
                    "reference_level": reference,
                    "level_n": int(
                        category_counts.loc[level]
                    ),
                    "level_share": float(
                        category_counts.loc[level]
                        / len(data)
                    ),
                    "ebm_reference_centered_log_odds": float(
                        ebm_score
                    ),
                    "logistic_reference_centered_log_odds": float(
                        logistic_score
                    ),
                    "ebm_minus_logistic_log_odds": float(
                        ebm_score
                        - logistic_score
                    ),
                }
            )

categorical_shape_functions = pd.DataFrame(
    categorical_shape_rows
)

categorical_shape_functions.to_csv(
    TABLE_DIR / "ebm_categorical_shape_functions.csv",
    index=False,
)

categorical_shape_functions


## 17. Continuous differences between target-specific EBM shapes

The difference is defined as

$$
\Delta_j(x)
=
f_j^{\mathrm{HbA1c}}(x)
-
f_j^{\mathrm{prior\ diagnosis}}(x),
$$

after both functions have been centred at the same reference value.

In [ ]:
continuous_prior = (
    continuous_shape_functions
    .loc[
        continuous_shape_functions["target"]
        == "self_reported_prior_diagnosis",
        [
            "variable",
            "variable_display_name",
            "value",
            "reference_value",
            "ebm_reference_centered_log_odds",
        ],
    ]
    .rename(
        columns={
            "ebm_reference_centered_log_odds": (
                "prior_diagnosis_ebm_log_odds"
            )
        }
    )
)

continuous_hba1c = (
    continuous_shape_functions
    .loc[
        continuous_shape_functions["target"]
        == "current_hba1c_ge_6_5",
        [
            "variable",
            "value",
            "ebm_reference_centered_log_odds",
        ],
    ]
    .rename(
        columns={
            "ebm_reference_centered_log_odds": (
                "hba1c_ebm_log_odds"
            )
        }
    )
)

continuous_cross_target_differences = (
    continuous_prior
    .merge(
        continuous_hba1c,
        on=["variable", "value"],
        how="inner",
        validate="one_to_one",
    )
)

continuous_cross_target_differences[
    "difference_hba1c_minus_prior_log_odds"
] = (
    continuous_cross_target_differences[
        "hba1c_ebm_log_odds"
    ]
    - continuous_cross_target_differences[
        "prior_diagnosis_ebm_log_odds"
    ]
)

continuous_cross_target_differences.to_csv(
    TABLE_DIR
    / "ebm_cross_target_continuous_shape_differences.csv",
    index=False,
)

continuous_cross_target_differences


## 18. Categorical differences between target-specific EBM contributions

In [ ]:
categorical_prior = (
    categorical_shape_functions
    .loc[
        categorical_shape_functions["target"]
        == "self_reported_prior_diagnosis",
        [
            "variable",
            "variable_display_name",
            "level",
            "reference_level",
            "level_n",
            "level_share",
            "ebm_reference_centered_log_odds",
        ],
    ]
    .rename(
        columns={
            "ebm_reference_centered_log_odds": (
                "prior_diagnosis_ebm_log_odds"
            )
        }
    )
)

categorical_hba1c = (
    categorical_shape_functions
    .loc[
        categorical_shape_functions["target"]
        == "current_hba1c_ge_6_5",
        [
            "variable",
            "level",
            "ebm_reference_centered_log_odds",
        ],
    ]
    .rename(
        columns={
            "ebm_reference_centered_log_odds": (
                "hba1c_ebm_log_odds"
            )
        }
    )
)

categorical_cross_target_differences = (
    categorical_prior
    .merge(
        categorical_hba1c,
        on=["variable", "level"],
        how="inner",
        validate="one_to_one",
    )
)

categorical_cross_target_differences[
    "difference_hba1c_minus_prior_log_odds"
] = (
    categorical_cross_target_differences[
        "hba1c_ebm_log_odds"
    ]
    - categorical_cross_target_differences[
        "prior_diagnosis_ebm_log_odds"
    ]
)

categorical_cross_target_differences.to_csv(
    TABLE_DIR
    / "ebm_cross_target_categorical_shape_differences.csv",
    index=False,
)

categorical_cross_target_differences


## Qualitative comparison with logistic regression

## 19. Continuous functional-form comparison

The metrics below summarise the descriptive distance between the EBM curve and the corresponding logistic straight line on the displayed grid. They are not hypothesis tests and do not determine which model is correct.

In [ ]:
continuous_comparison_rows = []

for (
    target,
    variable
), group in continuous_shape_functions.groupby(
    ["target", "variable"],
    observed=True,
):
    ebm_values = group[
        "ebm_reference_centered_log_odds"
    ].to_numpy()

    logistic_values = group[
        "logistic_reference_centered_log_odds"
    ].to_numpy()

    difference = (
        ebm_values
        - logistic_values
    )

    if (
        np.std(ebm_values) > 0
        and np.std(logistic_values) > 0
    ):
        correlation = float(
            np.corrcoef(
                ebm_values,
                logistic_values,
            )[0, 1]
        )
    else:
        correlation = np.nan

    continuous_comparison_rows.append(
        {
            "target": target,
            "target_display_name": (
                TARGET_DISPLAY_NAMES[target]
            ),
            "variable": variable,
            "variable_display_name": (
                CONTINUOUS_DISPLAY_NAMES[
                    variable
                ]
            ),
            "grid_n": int(
                len(group)
            ),
            "ebm_log_odds_range": float(
                ebm_values.max()
                - ebm_values.min()
            ),
            "logistic_log_odds_range": float(
                logistic_values.max()
                - logistic_values.min()
            ),
            "rmse_ebm_vs_logistic_log_odds": float(
                np.sqrt(
                    np.mean(
                        difference ** 2
                    )
                )
            ),
            "maximum_absolute_deviation_log_odds": float(
                np.max(
                    np.abs(difference)
                )
            ),
            "correlation_across_grid": (
                correlation
            ),
        }
    )

continuous_functional_form_comparison = pd.DataFrame(
    continuous_comparison_rows
)

continuous_functional_form_comparison.to_csv(
    TABLE_DIR
    / "ebm_vs_logistic_continuous_functional_form.csv",
    index=False,
)

continuous_functional_form_comparison


## 20. Categorical EBM-versus-logistic comparison

In [ ]:
categorical_functional_form_comparison = (
    categorical_shape_functions
    .loc[
        categorical_shape_functions["level"]
        != categorical_shape_functions[
            "reference_level"
        ],
        [
            "target",
            "target_display_name",
            "variable",
            "variable_display_name",
            "level",
            "reference_level",
            "level_n",
            "level_share",
            "ebm_reference_centered_log_odds",
            "logistic_reference_centered_log_odds",
            "ebm_minus_logistic_log_odds",
        ],
    ]
    .copy()
)

categorical_functional_form_comparison[
    "absolute_ebm_minus_logistic_log_odds"
] = (
    categorical_functional_form_comparison[
        "ebm_minus_logistic_log_odds"
    ].abs()
)

categorical_functional_form_comparison.to_csv(
    TABLE_DIR
    / "ebm_vs_logistic_categorical_functional_form.csv",
    index=False,
)

categorical_functional_form_comparison


### Interpretation rule

A visible difference between an EBM curve and a logistic straight line suggests that the additive nonlinear model learned a different functional form. It does not establish a causal threshold, biological mechanism, or clinically optimal cut-off.

Because the models use the same participants and predictors, these plots isolate differences in model functional form more clearly than comparisons based on different samples.

## Figure 3 — Target-specific EBM shape functions

## 21. Plotting helper for continuous EBM shapes

In [ ]:
def plot_ebm_target_shapes(
    variable: str,
    filename_stem: str,
    figure_title: str,
) -> tuple[Path, Path]:
    plot_data = (
        continuous_shape_functions
        .loc[
            continuous_shape_functions[
                "variable"
            ]
            == variable
        ]
        .copy()
    )

    figure, axis = plt.subplots(
        figsize=(9, 5.5)
    )

    for target in TARGET_COLUMNS:
        target_data = (
            plot_data
            .loc[
                plot_data["target"]
                == target
            ]
            .sort_values("value")
        )

        axis.plot(
            target_data["value"],
            target_data[
                "ebm_reference_centered_log_odds"
            ],
            label=TARGET_SHORT_NAMES[target],
        )

    reference = (
        CONTINUOUS_REFERENCE_VALUES[
            variable
        ]
    )

    axis.axhline(
        0,
        linewidth=1,
        linestyle="--",
    )

    axis.axvline(
        reference,
        linewidth=1,
        linestyle=":",
        label=f"Reference = {reference:g}",
    )

    axis.set_xlabel(
        CONTINUOUS_DISPLAY_NAMES[
            variable
        ]
    )

    axis.set_ylabel(
        "EBM contribution relative to reference "
        "(log-odds)"
    )

    axis.set_title(figure_title)
    axis.legend()

    figure.tight_layout()

    png_path = (
        FIGURE_DIR
        / f"{filename_stem}.png"
    )

    pdf_path = (
        FIGURE_DIR
        / f"{filename_stem}.pdf"
    )

    figure.savefig(
        png_path,
        dpi=300,
        bbox_inches="tight",
    )

    figure.savefig(
        pdf_path,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(figure)

    return png_path, pdf_path


## 22. Figure 3a: age

In [ ]:
figure3a_png_path, figure3a_pdf_path = (
    plot_ebm_target_shapes(
        variable="age",
        filename_stem=(
            "figure3a_ebm_age_shape_functions"
        ),
        figure_title=(
            "Figure 3a. EBM age shape functions"
        ),
    )
)

print("Saved:")
print(figure3a_png_path)
print(figure3a_pdf_path)


## 23. Figure 3b: BMI

In [ ]:
figure3b_png_path, figure3b_pdf_path = (
    plot_ebm_target_shapes(
        variable="bmi",
        filename_stem=(
            "figure3b_ebm_bmi_shape_functions"
        ),
        figure_title=(
            "Figure 3b. EBM BMI shape functions"
        ),
    )
)

print("Saved:")
print(figure3b_png_path)
print(figure3b_pdf_path)


## 24. Figure 3c: income-to-poverty ratio

In [ ]:
figure3c_png_path, figure3c_pdf_path = (
    plot_ebm_target_shapes(
        variable="income_poverty_ratio",
        filename_stem=(
            "figure3c_ebm_income_shape_functions"
        ),
        figure_title=(
            "Figure 3c. EBM income-to-poverty ratio "
            "shape functions"
        ),
    )
)

print("Saved:")
print(figure3c_png_path)
print(figure3c_pdf_path)


## 25. Appendix categorical EBM shape plots

In [ ]:
def plot_categorical_ebm_shapes(
    variable: str,
    filename_stem: str,
) -> tuple[Path, Path]:
    plot_data = (
        categorical_shape_functions
        .loc[
            categorical_shape_functions[
                "variable"
            ]
            == variable
        ]
        .copy()
    )

    levels = CATEGORY_LEVELS[variable]
    level_positions = np.arange(
        len(levels)
    )

    target_offsets = {
        TARGET_COLUMNS[0]: -0.12,
        TARGET_COLUMNS[1]: 0.12,
    }

    figure, axis = plt.subplots(
        figsize=(10, 5.5)
    )

    for target in TARGET_COLUMNS:
        target_data = (
            plot_data
            .loc[
                plot_data["target"]
                == target
            ]
            .set_index("level")
            .reindex(levels)
        )

        axis.plot(
            target_data[
                "ebm_reference_centered_log_odds"
            ],
            level_positions
            + target_offsets[target],
            linestyle="none",
            marker=target_markers[target],
            label=TARGET_SHORT_NAMES[target],
        )

    axis.axvline(
        0,
        linewidth=1,
        linestyle="--",
    )

    axis.set_yticks(
        level_positions,
        labels=levels,
    )

    axis.set_xlabel(
        "EBM contribution relative to reference "
        "(log-odds)"
    )

    axis.set_ylabel(
        CATEGORICAL_DISPLAY_NAMES[
            variable
        ]
    )

    axis.set_title(
        "Appendix. EBM categorical shape: "
        f"{CATEGORICAL_DISPLAY_NAMES[variable]}"
    )

    axis.legend()
    axis.invert_yaxis()

    figure.tight_layout()

    png_path = (
        FIGURE_DIR
        / f"{filename_stem}.png"
    )

    pdf_path = (
        FIGURE_DIR
        / f"{filename_stem}.pdf"
    )

    figure.savefig(
        png_path,
        dpi=300,
        bbox_inches="tight",
    )

    figure.savefig(
        pdf_path,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(figure)

    return png_path, pdf_path


categorical_figure_paths = {}

for variable in CATEGORICAL_PREDICTORS:
    stem = (
        "appendix_ebm_categorical_shape_"
        + variable
    )

    png_path, pdf_path = (
        plot_categorical_ebm_shapes(
            variable=variable,
            filename_stem=stem,
        )
    )

    categorical_figure_paths[
        f"{variable}_png"
    ] = png_path

    categorical_figure_paths[
        f"{variable}_pdf"
    ] = pdf_path

categorical_figure_paths


## Appendix — EBM versus logistic functional forms

## 26. Plotting helper

In [ ]:
def plot_ebm_vs_logistic(
    variable: str,
    filename_stem: str,
) -> tuple[Path, Path]:
    plot_data = (
        continuous_shape_functions
        .loc[
            continuous_shape_functions[
                "variable"
            ]
            == variable
        ]
        .copy()
    )

    figure, axis = plt.subplots(
        figsize=(9, 5.5)
    )

    for target in TARGET_COLUMNS:
        target_data = (
            plot_data
            .loc[
                plot_data["target"]
                == target
            ]
            .sort_values("value")
        )

        ebm_line = axis.plot(
            target_data["value"],
            target_data[
                "ebm_reference_centered_log_odds"
            ],
            linestyle="-",
            label=(
                f"EBM — "
                f"{TARGET_SHORT_NAMES[target]}"
            ),
        )[0]

        axis.plot(
            target_data["value"],
            target_data[
                "logistic_reference_centered_log_odds"
            ],
            linestyle="--",
            color=ebm_line.get_color(),
            label=(
                f"Logistic — "
                f"{TARGET_SHORT_NAMES[target]}"
            ),
        )

    reference = (
        CONTINUOUS_REFERENCE_VALUES[
            variable
        ]
    )

    axis.axhline(
        0,
        linewidth=1,
        linestyle="--",
    )

    axis.axvline(
        reference,
        linewidth=1,
        linestyle=":",
    )

    axis.set_xlabel(
        CONTINUOUS_DISPLAY_NAMES[
            variable
        ]
    )

    axis.set_ylabel(
        "Contribution relative to reference "
        "(log-odds)"
    )

    axis.set_title(
        "Appendix. EBM and logistic functional forms: "
        f"{CONTINUOUS_DISPLAY_NAMES[variable]}"
    )

    axis.legend()

    figure.tight_layout()

    png_path = (
        FIGURE_DIR
        / f"{filename_stem}.png"
    )

    pdf_path = (
        FIGURE_DIR
        / f"{filename_stem}.pdf"
    )

    figure.savefig(
        png_path,
        dpi=300,
        bbox_inches="tight",
    )

    figure.savefig(
        pdf_path,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(figure)

    return png_path, pdf_path


## 27. Create EBM-versus-logistic comparison plots

In [ ]:
functional_form_figure_paths = {}

for variable in CONTINUOUS_PREDICTORS:
    stem = (
        "appendix_ebm_vs_logistic_"
        + variable
    )

    png_path, pdf_path = (
        plot_ebm_vs_logistic(
            variable=variable,
            filename_stem=stem,
        )
    )

    functional_form_figure_paths[
        f"{variable}_png"
    ] = png_path

    functional_form_figure_paths[
        f"{variable}_pdf"
    ] = pdf_path

functional_form_figure_paths


## Compact paper and appendix tables

## 28. Feature-importance comparison across targets

In [ ]:
paper_importance = (
    global_feature_importance
    .pivot(
        index=[
            "term",
            "display_name",
        ],
        columns="target",
        values=[
            "normalised_importance_share",
            "within_target_rank",
        ],
    )
)

paper_importance.columns = [
    "_".join(
        str(part)
        for part in column
    )
    for column in paper_importance.columns
]

paper_importance = (
    paper_importance
    .reset_index()
)

paper_importance.to_csv(
    TABLE_DIR
    / "paper_ebm_global_importance_comparison.csv",
    index=False,
)

paper_importance


## 29. Shape-comparison summary for drafting

In [ ]:
paper_shape_summary = (
    continuous_functional_form_comparison
    [
        [
            "target_display_name",
            "variable_display_name",
            "ebm_log_odds_range",
            "logistic_log_odds_range",
            "rmse_ebm_vs_logistic_log_odds",
            "maximum_absolute_deviation_log_odds",
            "correlation_across_grid",
        ]
    ]
    .copy()
)

paper_shape_summary.to_csv(
    TABLE_DIR
    / "paper_ebm_shape_comparison_summary.csv",
    index=False,
)

paper_shape_summary


## Save metadata and final checkpoint

In [ ]:
def json_safe(value):
    if isinstance(value, Path):
        return str(value)

    if isinstance(
        value,
        (
            np.integer,
            np.floating,
        ),
    ):
        return value.item()

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, dict):
        return {
            str(key): json_safe(item)
            for key, item in value.items()
        }

    if isinstance(
        value,
        (list, tuple),
    ):
        return [
            json_safe(item)
            for item in value
        ]

    return value


output_files = {
    "fixed_fold_assignments": (
        FOLD_ASSIGNMENT_PATH
    ),
    "ebm_oof_predictions": (
        EBM_OOF_PREDICTIONS_PATH
    ),
    "category_fold_coverage": (
        TABLE_DIR
        / "ebm_category_fold_coverage.csv"
    ),
    "fold_fit_summary": (
        TABLE_DIR / "ebm_fold_fit_summary.csv"
    ),
    "fold_feature_importance": (
        TABLE_DIR
        / "ebm_fold_feature_importance.csv"
    ),
    "full_fit_summary": (
        TABLE_DIR / "ebm_full_fit_summary.csv"
    ),
    "global_feature_importance": (
        TABLE_DIR
        / "ebm_global_feature_importance.csv"
    ),
    "continuous_shape_support": (
        TABLE_DIR
        / "ebm_continuous_shape_support.csv"
    ),
    "continuous_shape_functions": (
        TABLE_DIR
        / "ebm_continuous_shape_functions.csv"
    ),
    "categorical_shape_functions": (
        TABLE_DIR
        / "ebm_categorical_shape_functions.csv"
    ),
    "continuous_cross_target_differences": (
        TABLE_DIR
        / "ebm_cross_target_continuous_shape_differences.csv"
    ),
    "categorical_cross_target_differences": (
        TABLE_DIR
        / "ebm_cross_target_categorical_shape_differences.csv"
    ),
    "continuous_functional_form_comparison": (
        TABLE_DIR
        / "ebm_vs_logistic_continuous_functional_form.csv"
    ),
    "categorical_functional_form_comparison": (
        TABLE_DIR
        / "ebm_vs_logistic_categorical_functional_form.csv"
    ),
    "importance_png": importance_png_path,
    "importance_pdf": importance_pdf_path,
    "figure3a_png": figure3a_png_path,
    "figure3a_pdf": figure3a_pdf_path,
    "figure3b_png": figure3b_png_path,
    "figure3b_pdf": figure3b_pdf_path,
    "figure3c_png": figure3c_png_path,
    "figure3c_pdf": figure3c_pdf_path,
    "prior_diagnosis_ebm_joblib": (
        EBM_MODEL_PATHS[
            "self_reported_prior_diagnosis"
        ]
    ),
    "hba1c_ebm_joblib": (
        EBM_MODEL_PATHS[
            "current_hba1c_ge_6_5"
        ]
    ),
    "prior_diagnosis_ebm_json": (
        EBM_JSON_PATHS[
            "self_reported_prior_diagnosis"
        ]
    ),
    "hba1c_ebm_json": (
        EBM_JSON_PATHS[
            "current_hba1c_ge_6_5"
        ]
    ),
    **{
        f"categorical_{name}": path
        for name, path
        in categorical_figure_paths.items()
    },
    **{
        f"functional_form_{name}": path
        for name, path
        in functional_form_figure_paths.items()
    },
}

missing_output_files = [
    str(path)
    for path in output_files.values()
    if not path.exists()
]

if missing_output_files:
    raise FileNotFoundError(
        "The following expected EBM outputs are missing:\n"
        + "\n".join(missing_output_files)
    )

ebm_metadata = {
    "analytic_sample_n": int(
        len(data)
    ),
    "analysis_id_sha256": (
        analysis_id_hash
    ),
    "random_state": RANDOM_STATE,
    "n_splits": N_SPLITS,
    "fold_assignment_path": str(
        FOLD_ASSIGNMENT_PATH
    ),
    "folds_reused_from_notebook04": True,
    "predictor_columns": (
        PREDICTOR_COLUMNS
    ),
    "feature_types": FEATURE_TYPES,
    "continuous_predictors": (
        CONTINUOUS_PREDICTORS
    ),
    "categorical_predictors": (
        CATEGORICAL_PREDICTORS
    ),
    "reference_categories_for_plots": (
        REFERENCE_CATEGORIES
    ),
    "continuous_reference_values": (
        CONTINUOUS_REFERENCE_VALUES
    ),
    "target_columns": TARGET_COLUMNS,
    "oof_probability_columns": (
        OOF_PROBABILITY_COLUMNS
    ),
    "ebm_parameters": json_safe(
        EBM_PARAMETERS
    ),
    "interactions_disabled": True,
    "class_weighting": None,
    "sample_weighting": None,
    "shape_scale": (
        "Reference-centred additive contribution "
        "on the log-odds scale"
    ),
    "bmi_plot_support": (
        "1st to 99th observed percentile; "
        "all observations retained in fitting"
    ),
    "software_versions": {
        "python": (
            platform.python_version()
        ),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": (
            sklearn.__version__
        ),
        "interpret": (
            interpret.__version__
        ),
        "joblib": joblib.__version__,
    },
    "output_files": {
        name: str(path)
        for name, path
        in output_files.items()
    },
}

with EBM_METADATA_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        ebm_metadata,
        file,
        indent=2,
    )

final_checkpoint = {
    "analytic_sample_n": int(
        len(data)
    ),
    "same_analysis_hash_as_logistic": bool(
        analysis_id_hash
        == logistic_metadata[
            "analysis_id_sha256"
        ]
    ),
    "same_predictors_as_logistic": bool(
        logistic_metadata[
            "predictor_columns"
        ]
        == PREDICTOR_COLUMNS
    ),
    "fixed_five_folds_reused": bool(
        set(data["cv_fold"])
        == set(range(1, N_SPLITS + 1))
    ),
    "all_training_folds_contain_all_categories": bool(
        category_fold_coverage[
            "training_n"
        ].gt(0).all()
    ),
    "missing_oof_probabilities": int(
        oof_predictions[
            list(
                OOF_PROBABILITY_COLUMNS.values()
            )
        ]
        .isna()
        .sum()
        .sum()
    ),
    "oof_probabilities_within_unit_interval": bool(
        all(
            oof_predictions[column]
            .between(0, 1)
            .all()
            for column
            in OOF_PROBABILITY_COLUMNS.values()
        )
    ),
    "full_models_are_strictly_additive": bool(
        all(
            all(
                len(term) == 1
                for term
                in model.term_features_
            )
            for model
            in full_ebm_models.values()
        )
    ),
    "main_terms_per_model": {
        target: int(
            len(model.term_names_)
        )
        for target, model
        in full_ebm_models.items()
    },
    "continuous_shape_rows": int(
        len(
            continuous_shape_functions
        )
    ),
    "categorical_shape_rows": int(
        len(
            categorical_shape_functions
        )
    ),
    "all_expected_outputs_saved": bool(
        len(missing_output_files) == 0
    ),
    "metadata_saved": bool(
        EBM_METADATA_PATH.exists()
    ),
}

print("Saved EBM metadata to:")
print(EBM_METADATA_PATH)
print()
print("Final checkpoint:")
final_checkpoint


## Completion criteria

- EBM interactions remain disabled.
- The exact Notebook 04 sample and folds are reused.
- All model, shape, figure, and metadata outputs are written.